# 1. Загрузка, аудит и очистка данных

**Проект:** прогноз повторной госпитализации пациентов с диабетом (тема 39).
**Задача:** по данным одной госпитализации предсказать, вернётся ли пациент в больницу в течение 30 дней.

Это первый из трёх ноутбуков:

| Ноутбук | Что показывает |
|---|---|
| **01_data_cleaning** (этот) | загрузка, аудит качества, очистка, новые признаки |
| 02_eda | статистика, выбросы, графики |
| 03_models | разбиение, модели, метрики, калибровка, прогноз |

В ноутбуке нет собственной логики: все функции лежат в `src/`, здесь мы только вызываем их и смотрим на результат.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))   # чтобы работал импорт from src ...

import pandas as pd
from IPython.display import Image, display

from src import config, data_loader, preprocessing

pd.set_option('display.max_columns', 60)

## 1.1. Загрузка

В этом датасете пропуски записаны знаком `?`, а не пустой ячейкой. Если прочитать файл как обычно, pandas посчитает `?` настоящим значением и пропусков «не будет». Поэтому `load_raw` читает с `na_values="?"`.

In [2]:
raw = data_loader.load_raw()
print('Размер таблицы:', raw.shape)
raw.head(3)

Размер таблицы: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO


Одна строка — одна госпитализация (`encounter_id`). Один пациент (`patient_nbr`) может встречаться несколько раз.

### Память

Текстовые столбцы с малым числом разных значений выгодно хранить как `category`, а числа — в самом маленьком подходящем типе. Это делает `optimize_dtypes`. Тот же файл можно читать кусками (`load_in_chunks`): тогда в памяти одновременно лежит только один «сырой» кусок, и подход работает даже для файлов больше оперативной памяти.

In [3]:
optimized = data_loader.optimize_dtypes(raw)
chunked = data_loader.load_in_chunks(chunksize=20000)

print('Память до оптимизации:   ', data_loader.memory_mb(raw), 'МБ')
print('Память после оптимизации:', data_loader.memory_mb(optimized), 'МБ')
print('Чтение кусками:          ', data_loader.memory_mb(chunked), 'МБ, строк', len(chunked))

Память до оптимизации:    48.51 МБ
Память после оптимизации: 5.85 МБ
Чтение кусками:           5.85 МБ, строк 101766


Таблица стала занимать примерно в 8 раз меньше памяти, данные при этом не изменились.

## 1.2. Аудит качества

Прежде чем что-то удалять, смотрим на данные **до** очистки. Функция `audit` ничего не меняет, только считает факты.

In [4]:
audit = preprocessing.audit(raw)
for key in ['rows', 'columns', 'unique_patients', 'patients_with_repeat_visits',
            'rows_from_repeat_patients', 'duplicate_encounter_ids', 'gender_unknown_rows']:
    print(f'{key:32} {audit[key]}')

rows                             101766
columns                          50
unique_patients                  71518
patients_with_repeat_visits      16773
rows_from_repeat_patients        47021
duplicate_encounter_ids          0
gender_unknown_rows              3


**Повторные пациенты.** 16 773 пациента приходили больше одного раза, и на них приходится 47 021 строка — это 46% всех данных. Отсюда главное правило проекта: делить данные на train и test нужно **по пациентам**, иначе один и тот же человек окажется в обеих частях (подробно — в ноутбуке 03).

Дубликатов `encounter_id` нет — каждая строка действительно отдельный визит.

### Пропуски

In [5]:
pd.Series(audit['missing_percent'], name='пропущено, %').to_frame()

,"пропущено, %"
weight,96.86
max_glu_serum,94.75
A1Cresult,83.28
medical_specialty,49.08
payer_code,39.56
race,2.23
diag_3,1.40
diag_2,0.35
diag_1,0.02


- `weight` пропущен почти везде (97%) — восстановить его нельзя, столбец удалим.
- `max_glu_serum` и `A1Cresult` пропущены потому, что анализ пациенту **не назначали**. Это не ошибка, а информация, поэтому пропуск превратим в отдельную категорию.
- `medical_specialty` и `payer_code` пропущены в 40–49% строк — удалять столько строк нельзя, заменим на `Unknown`.

### Целевая переменная

In [6]:
target = pd.Series(audit['target_distribution'], name='визитов').to_frame()
target['доля, %'] = (target['визитов'] / target['визитов'].sum() * 100).round(1)
target

,визитов,"доля, %"
NO,54864,53.9
>30,35545,34.9
<30,11357,11.2


Раньше 30 дней вернулись только ~11% — классы сильно несбалансированы. Это учитывается при обучении моделей и выборе метрик.

### Умершие и выписанные в хоспис

В справочнике `IDS_mapping.csv` лежат расшифровки кодов. В одном файле там три таблицы подряд, поэтому `load_mapping` читает его построчно и возвращает словарь из трёх таблиц.

In [7]:
mapping = data_loader.load_mapping()
for name, table in mapping.items():
    print(f'{name:28} {len(table)} кодов')

discharge = mapping['discharge_disposition_id']
display(discharge[discharge['id'].isin(config.EXPIRED_OR_HOSPICE)])

print('Строк «умер / хоспис»:           ', audit['expired_or_hospice_rows'])
print('Из них записаны как вернувшиеся: ', audit['expired_but_marked_readmitted'])

admission_type_id            8 кодов
discharge_disposition_id     30 кодов
admission_source_id          25 кодов


,id,description
10,11,Expired
12,13,Hospice / home
13,14,Hospice / medical facility
18,19,"Expired at home. Medicaid only, hospice."
19,20,"Expired in a medical facility. Medicaid only, ..."
20,21,"Expired, place unknown. Medicaid only, hospice."


Строк «умер / хоспис»:            2423
Из них записаны как вернувшиеся:  86


2 423 визита закончились смертью или переводом в хоспис. Такой пациент вернуться в больницу не может. Если оставить эти строки, модель выучит бесполезное правило «умер → не вернётся». При этом 86 таких строк помечены как «вернулся» — это ошибка в исходных данных, ещё одна причина их убрать.

## 1.3. Очистка

Функция `clean` применяет девять правил по порядку:

| № | Правило | Почему |
|---|---|---|
| 1 | удалить визиты «умер / хоспис» | вернуться не могут |
| 2 | удалить `gender == "Unknown/Invalid"` | всего 3 строки, мусорная категория |
| 3 | удалить `weight` и почти постоянные столбцы | не несут информации |
| 4 | `A1Cresult`, `max_glu_serum`: пропуск → `NotMeasured` | «анализ не делали» — это информация |
| 5 | `medical_specialty`, `payer_code`, `race`: пропуск → `Unknown` | пропусков слишком много, чтобы удалять строки |
| 6 | коды диагнозов ICD-9 → группы | сотни редких кодов модель не выучит |
| 7 | `readmit_30` = 1, если `readmitted == "<30"` | целевая переменная |
| 8 | `total_prior_visits`, `age_num` | новые признаки |
| 9 | перенумеровать строки | после удаления в индексе дыры |

In [8]:
clean = preprocessing.clean(raw)

print('Строк было: ', len(raw))
print('Строк стало:', len(clean), f'(удалено {len(raw) - len(clean)})')
print('Столбцов было / стало:', raw.shape[1], '/', clean.shape[1])

Строк было:  101766
Строк стало: 99340 (удалено 2426)
Столбцов было / стало: 50 / 44


In [9]:
removed = [col for col in raw.columns if col not in clean.columns]
added = [col for col in clean.columns if col not in raw.columns]
print('Удалённые столбцы:', len(removed))
print(removed)
print()
print('Новые столбцы:', len(added))
print(added)

Удалённые столбцы: 12
['weight', 'acetohexamide', 'tolbutamide', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']

Новые столбцы: 6
['diag_1_group', 'diag_2_group', 'diag_3_group', 'readmit_30', 'total_prior_visits', 'age_num']


**Почти постоянные столбцы.** `find_near_constant` ищет столбцы, где одно значение занимает больше 99.95% строк. Нашлось 11 лекарств, которые почти никому не назначали. Пример — `examide`: во всех 101 766 строках стоит `No`, столбец не отличает одного пациента от другого.

In [10]:
print('Почти постоянные столбцы:', preprocessing.find_near_constant(raw))
print()
print(raw['examide'].value_counts())

Почти постоянные столбцы: ['acetohexamide', 'tolbutamide', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']

examide
No    101766
Name: count, dtype: int64


Пропусков в столбцах, которые идут в модель, после очистки не осталось:

In [11]:
numeric, categorical = preprocessing.get_feature_lists(clean)
print('Пропусков в признаках модели:', int(clean[numeric + categorical].isna().sum().sum()))
clean[['A1Cresult', 'max_glu_serum', 'medical_specialty', 'payer_code', 'race']].apply(lambda col: col.value_counts().head(3).to_dict())

Пропусков в признаках модели: 0


A1Cresult             {'NotMeasured': 82506, '>8': 8137, 'Norm': 4922}
max_glu_serum        {'NotMeasured': 94188, 'Norm': 2545, '>200': 1...
medical_specialty    {'Unknown': 48614, 'InternalMedicine': 14237, ...
payer_code                 {'Unknown': 39397, 'MC': 31333, 'HM': 6195}
race                 {'Caucasian': 74220, 'AfricanAmerican': 18772,...
dtype: object

## 1.4. Новые признаки

### Группы диагнозов (ICD-9)

У диагноза около 700 разных кодов, многие встречаются по несколько раз. `map_icd9` сводит их к 9 группам болезней по схеме Strack et al., 2014 (плюс `Missing` для пропуска).

In [12]:
for code in ['428', '250.83', '486', 'V57', '820', None]:
    print(f'{str(code):8} -> {preprocessing.map_icd9(code)}')

428      -> Circulatory
250.83   -> Diabetes
486      -> Respiratory
V57      -> Other
820      -> Injury
None     -> Missing


In [13]:
print('Разных кодов основного диагноза:', raw['diag_1'].nunique())
print('Групп после преобразования:     ', clean['diag_1_group'].nunique())
clean['diag_1_group'].value_counts().to_frame('визитов')

Разных кодов основного диагноза: 716
Групп после преобразования:      10


,визитов
diag_1_group,
Circulatory,29680
Other,17793
Respiratory,13934
Digestive,9333
Diabetes,8661
Injury,6851
Genitourinary,5002
Musculoskeletal,4935
Neoplasms,3131


### Возраст числом и общее число прошлых обращений

- `age_num` — середина возрастного интервала: `[70-80)` → 75. Модели проще работать с числом, чем с текстом интервала.
- `total_prior_visits` — сумма амбулаторных визитов, визитов в скорую и госпитализаций за прошлый год.

In [14]:
clean[['age', 'age_num', 'number_outpatient', 'number_emergency', 'number_inpatient', 'total_prior_visits']].head(5)

,age,age_num,number_outpatient,number_emergency,number_inpatient,total_prior_visits
0,[0-10),5,0,0,0,0
1,[10-20),15,0,0,0,0
2,[20-30),25,2,0,1,3
3,[30-40),35,0,0,0,0
4,[40-50),45,0,0,0,0


### Целевая переменная `readmit_30`

Из трёх исходных значений делаем два: 1 — вернулся раньше 30 дней, 0 — всё остальное. Возврат позже 30 дней считаем нулём, потому что задача именно про ранний возврат.

In [15]:
pd.crosstab(clean['readmitted'], clean['readmit_30'])

readmit_30,0,1
readmitted,,
<30,0,11314
>30,35502,0
NO,52524,0


In [16]:
print('Доля вернувшихся раньше 30 дней:', round(clean['readmit_30'].mean() * 100, 1), '%')

Доля вернувшихся раньше 30 дней: 11.4 %


## 1.5. Что идёт в модель

36 признаков: 10 числовых и 26 категориальных.

In [17]:
print('Числовые:', len(numeric))
print(numeric)
print()
print('Категориальные:', len(categorical))
print(categorical)

Числовые: 10
['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'total_prior_visits', 'age_num']

Категориальные: 26
['race', 'gender', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'medical_specialty', 'payer_code', 'A1Cresult', 'max_glu_serum', 'diag_1_group', 'diag_2_group', 'diag_3_group', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'insulin', 'glyburide-metformin', 'change', 'diabetesMed']


In [18]:
not_used = [col for col in clean.columns if col not in numeric + categorical]
print('В модель НЕ идут:', not_used)

В модель НЕ идут: ['encounter_id', 'patient_nbr', 'age', 'diag_1', 'diag_2', 'diag_3', 'readmitted', 'readmit_30']


- `encounter_id`, `patient_nbr` — это номера, а не свойства пациента (`patient_nbr` нужен только для разбиения на train/test);
- `readmitted` — из него сделан ответ, оставить его значит подсказать модели ответ (утечка);
- `diag_1..3` и `age` — заменены на группы и `age_num`;
- `readmit_30` — сам ответ.

## 1.6. Сохранение

Очищенную таблицу сохраняем в Parquet: он хранит типы столбцов и занимает намного меньше места, чем CSV.

In [19]:
config.ensure_dirs()
data_loader.optimize_dtypes(clean).to_parquet(config.CLEAN_PARQUET, index=False)

print('Сохранено:', config.CLEAN_PARQUET.relative_to(config.ROOT))
print('Исходный CSV:', round(config.RAW_CSV.stat().st_size / 1024**2, 1), 'МБ')
print('Parquet:     ', round(config.CLEAN_PARQUET.stat().st_size / 1024**2, 1), 'МБ')

Сохранено: data\processed\clean.parquet
Исходный CSV: 18.3 МБ
Parquet:      2.6 МБ


## Итог

1. В данных 101 766 визитов и 71 518 пациентов; 46% строк принадлежат пациентам с несколькими визитами.
2. Пропуски были записаны знаком `?`; больше всего их в `weight` (97%), `max_glu_serum` (95%) и `A1Cresult` (83%).
3. Удалено 2 426 строк: 2 423 визита «умер / хоспис» и 3 строки без пола. Осталось 99 340.
4. Удалено 12 столбцов: `weight` и 11 лекарств, которые почти никому не назначали.
5. Добавлено 6 столбцов: три группы диагнозов, `age_num`, `total_prior_visits` и целевая `readmit_30` (11.4% единиц).